In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
ENTREPOT_PATH = "/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/"
donnees = {}

def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables = [
    'composant_culture',
    'culture',
    # 'espece',
    # 'recolte_rendement_prix',
    # 'recolte_rendement_prix_restructure'
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 2/2 [00:03<00:00,  1.74s/it]


In [3]:
matrice_passage_dirodur_espece_precise= pd.read_csv("/home/administrateur/Bureau/"+"matrice_passage_dirodur_espece_precise.csv", sep=',')
matrice_passage_dirodur_espece_famille_bota = pd.read_csv("/home/administrateur/Bureau/"+"matrice_passage_dirodur_famille_bota.csv", sep=',')
sp = pd.read_csv("/home/administrateur/Bureau/"+"ref_espece_gcpe.csv", sep=',')

In [4]:
# def get_typologie_culture_DIRODUR(donnees):
#     ''' 
#     Le but est d'obtenir les typologies d'espece et de culture utilisées par DIRODUR.

#     Note(s):
#         La typologie espece a été directement intégré au référentiel refespece d'Agrosyst

#     Echelle :
#         culture_id

#     Args:
#         donnees (dict):
#             Données d'entrepot
#             - 'composant_culture'
#             - 'culture'
#             - 'espece'
#             - 'recolte_rendement_prix'
#             - 'recolte_rendement_prix_restructure'
#             Données externe (référentiel CAN):
#             - 'matrice_passage_esp_culture_dirodur.csv'

#     Returns:
#         pd.DataFrame() contenant la culture_id et la typologie de culture de la CAN
#     '''

In [5]:
# Donnes de bases
# TODO : .reset_index() avant copy()) si le test marche pas
cropsp = donnees['composant_culture'].copy()
cropsp = cropsp[['id','espece_id','culture_id','compagne']].rename(columns={'id':'composant_culture_id'})
crop = donnees['culture'].copy()
crop = crop[['id','nom','type']].rename(columns={'id':'culture_id'})
# sp = donnees['espece'].copy()
sp = sp[['id','typodirodur_espece_precise','typodirodur_espece_famille_bota']].rename(columns={'id':'espece_id'})
# TODO matrice_passage = donnees['matrice_passage_esp_culture_dirodur'].copy()


df = cropsp.merge(sp, how = 'left', on = 'espece_id')
# Liste des cultures qui contiennent des cultures compagnes
list_culture_with_compagne = list(set(df.loc[df['compagne'].notnull(), 'culture_id']))

df['nb_composant_culture'] = 1
df['nb_typodirodur_espece_precise'] = df['typodirodur_espece_precise'].copy()
df['nb_typodirodur_espece_famille_bota'] = df['typodirodur_espece_famille_bota'].copy()

def concat_unique_sorted(series):
    cleaned = series.dropna().unique()
    if len(cleaned) == 0:
        return np.nan
    return '_'.join(sorted(cleaned))
def get_nb_unique_typo(series):
    cleaned = series.dropna().unique()
    return len(cleaned)
agg_dict = {
    'typodirodur_espece_precise': concat_unique_sorted,
    'typodirodur_espece_famille_bota': concat_unique_sorted,
    'nb_composant_culture': 'sum',
    'nb_typodirodur_espece_precise': get_nb_unique_typo,
    'nb_typodirodur_espece_famille_bota': get_nb_unique_typo,
}

#  On crée les typologie can culture et les autre variable utiles grace a agg_dict
df = df[['culture_id',
         'nb_composant_culture',
         'typodirodur_espece_precise',
         'typodirodur_espece_famille_bota',
         'nb_typodirodur_espece_precise',
         'nb_typodirodur_espece_famille_bota']].groupby('culture_id').agg(agg_dict).reset_index()

# On ajoute les type des culture_id
# On fait un outer pour avoir toutes les cultures meme celles qui n'ont pas de composant_culture
df = df.merge(crop[['culture_id','nom','type']], how='outer', on='culture_id')

# Détection des cultures qui contiennent des cultures compagnes
df['is_any_compagne'] = np.where(df['culture_id'].isin(list_culture_with_compagne), True, False)

df.loc[df['nb_composant_culture'].isna(),['nb_composant_culture','nb_typodirodur_espece_precise','nb_typodirodur_espece_famille_bota']] = 0

In [6]:
matrice_passage_dirodur_espece_precise

,typodirodur_espece_precise,culture_dirodur
0,Pois Fourrager / Fourrage Hiver_Triticale,267
1,Fétuque élevée_Ray-grass anglais_Trèfle blanc,242
2,Blé tendre Hiver Amidon_Blé tendre Hiver Meunier,202
3,Blé tendre Hiver Meunier_Fèverole Hiver,189
4,Ray-grass d'Italie_Trèfle incarnat,178
...,...,...
1735,Avoine Noir(e) Hiver_Pois Fourrager / Fourrage...,1
1736,Fèverole Printemps Grain(es)_Maïs amidon,1
1737,Dactyle_Fétuque des prés_Luzerne_Ray-grass d'I...,1
1738,Blé tendre Printemps Meunier_Cameline,1


In [7]:
matrice_passage_dirodur_espece_famille_bota

,typodirodur_espece_famille_bota,culture_dirodur_famille_bota
0,Fabaceae_Poaceae,6427
1,Brassicaceae_Fabaceae,423
2,Brassicaceae_Poaceae,50
3,Asteraceae_Fabaceae,41
4,Brassicaceae_Fabaceae_Polygonaceae,40
5,Asteraceae_Fabaceae_Poaceae,39
6,Brassicaceae_Fabaceae_Poaceae,37
7,Asteraceae_Poaceae,25
8,Fabaceae_Plantaginaceae_Poaceae,20
9,Brassicaceae_Polygonaceae,17


In [8]:
df

,culture_id,typodirodur_espece_precise,typodirodur_espece_famille_bota,nb_composant_culture,nb_typodirodur_espece_precise,nb_typodirodur_espece_famille_bota,nom,type,is_any_compagne
0,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Blé tendre Hiver Biscuit,Poaceae,1.0,1.0,1.0,Blé tendre,MAIN,False
1,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Blé tendre Hiver Amidon,Poaceae,4.0,1.0,1.0,Blé tendre d'hiver,MAIN,False
2,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Orge 6 rangs Hiver,Poaceae,1.0,1.0,1.0,Orge,MAIN,False
3,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,NaN,NaN,1.0,0.0,0.0,Fraise de printemps,MAIN,False
4,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Autre légumineuse_Mélange Floristique_Mélange ...,Fabaceae_Poaceae,3.0,3.0,2.0,Prairie Permanente,MAIN,False
...,...,...,...,...,...,...,...,...,...
251683,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Blé tendre Hiver Meunier,Poaceae,1.0,1.0,1.0,Blé tendre,MAIN,False
251684,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Colza Oléagineux Hiver,Brassicaceae,1.0,1.0,1.0,Colza,MAIN,False
251685,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,NaN,NaN,3.0,0.0,0.0,Plantes à massifs,MAIN,False
251686,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Colza Oléagineux Hiver,Brassicaceae,1.0,1.0,1.0,COLZA D'HIVER,MAIN,False


In [9]:
df = df.merge(matrice_passage_dirodur_espece_precise, how='left', on='typodirodur_espece_precise')
df = df.merge(matrice_passage_dirodur_espece_famille_bota, how='left', on='typodirodur_espece_famille_bota')

# S'il n'y a qu'une seule typologie, on prends cette typologie en tant que typo culture DIRODUR, sinon on prends celle qui a été déterminé grace à la matrice de passage
# S'il n'y a aucune typologie d'espèce précise de renseignée, on attribue l'erreur "Aucune_typologie_espece_XXX"
# S'il n'y a aucun composant de culture, on attribue la typologie "Aucun_composant_de_culture"
# L'ordre des conditions est important !!!
df['culture_dirodur'] = np.where(df['nb_typodirodur_espece_precise'] == 1, df['typodirodur_espece_precise'], df['culture_dirodur'])
df['culture_dirodur'] = np.where(df['nb_typodirodur_espece_precise'] == 0, "Aucune_typologie_espece_precise", df['culture_dirodur'])
df['culture_dirodur'] = np.where(df['nb_composant_culture'] == 0, "Aucun_composant_de_culture", df['culture_dirodur'])

df['culture_dirodur_famille_bota'] = np.where(df['nb_typodirodur_espece_famille_bota'] == 1, df['typodirodur_espece_famille_bota'], df['culture_dirodur_famille_bota'])
df['culture_dirodur_famille_bota'] = np.where(df['nb_typodirodur_espece_famille_bota'] == 0, "Aucune_typologie_espece_famille_bota", df['culture_dirodur_famille_bota'])
df['culture_dirodur_famille_bota'] = np.where(df['nb_composant_culture'] == 0, "Aucun_composant_de_culture", df['culture_dirodur_famille_bota'])

In [10]:
df['type'] = df['type'].astype('category')
df['type'] = df['type'].cat.rename_categories({'MAIN': 'PRINCIPALE', 
                                                'INTERMEDIATE': 'INTERMEDIAIRE', 
                                                'CATCH': 'DEROBEE' })
df['type'] = df['type'].astype('str')
df[['nb_composant_culture','nb_typodirodur_espece_precise','nb_typodirodur_espece_famille_bota']] = df[['nb_composant_culture','nb_typodirodur_espece_precise','nb_typodirodur_espece_famille_bota']].astype('int64')

In [11]:
df

,culture_id,typodirodur_espece_precise,typodirodur_espece_famille_bota,nb_composant_culture,nb_typodirodur_espece_precise,nb_typodirodur_espece_famille_bota,nom,type,is_any_compagne,culture_dirodur,culture_dirodur_famille_bota
0,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Blé tendre Hiver Biscuit,Poaceae,1,1,1,Blé tendre,PRINCIPALE,False,Blé tendre Hiver Biscuit,Poaceae
1,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Blé tendre Hiver Amidon,Poaceae,4,1,1,Blé tendre d'hiver,PRINCIPALE,False,Blé tendre Hiver Amidon,Poaceae
2,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Orge 6 rangs Hiver,Poaceae,1,1,1,Orge,PRINCIPALE,False,Orge 6 rangs Hiver,Poaceae
3,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,NaN,NaN,1,0,0,Fraise de printemps,PRINCIPALE,False,Aucune_typologie_espece_precise,Aucune_typologie_espece_famille_bota
4,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Autre légumineuse_Mélange Floristique_Mélange ...,Fabaceae_Poaceae,3,3,2,Prairie Permanente,PRINCIPALE,False,69.0,6427.0
...,...,...,...,...,...,...,...,...,...,...,...
251683,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Blé tendre Hiver Meunier,Poaceae,1,1,1,Blé tendre,PRINCIPALE,False,Blé tendre Hiver Meunier,Poaceae
251684,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Colza Oléagineux Hiver,Brassicaceae,1,1,1,Colza,PRINCIPALE,False,Colza Oléagineux Hiver,Brassicaceae
251685,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,NaN,NaN,3,0,0,Plantes à massifs,PRINCIPALE,False,Aucune_typologie_espece_precise,Aucune_typologie_espece_famille_bota
251686,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,Colza Oléagineux Hiver,Brassicaceae,1,1,1,COLZA D'HIVER,PRINCIPALE,False,Colza Oléagineux Hiver,Brassicaceae
